# Feature table analysis

This notebook analyzes `analysis/features.json` as a feature table (enums + notes).

It provides:
- Quick sanity checks
- Crosstabs / slices
- Optional clustering (one-hot features)


In [5]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd

# Robustly locate repo root even if notebook runs from /analysis
REPO_ROOT = Path.cwd().resolve()
if (REPO_ROOT / 'analysis' / 'features.json').exists():
  pass
elif (REPO_ROOT.parent / 'analysis' / 'features.json').exists():
  REPO_ROOT = REPO_ROOT.parent
else:
  raise FileNotFoundError('Could not locate analysis/features.json from current working directory')

FEATURES_PATH = REPO_ROOT / 'analysis' / 'features.json'

with FEATURES_PATH.open('r', encoding='utf-8') as f:
  data = json.load(f)

rows = data.get('rows', [])
print('repo_root', REPO_ROOT)
print('rows', len(rows))
assert rows, 'No rows found. Run scripts/extract_feature_table.py first.'


repo_root /Users/kihyvr/Documents/design-system-analysis
rows 58


In [6]:
# Flatten into a dataframe (schema-driven, fills missing keys with 'unknown')
schema_path = REPO_ROOT / 'analysis' / 'features.schema.json'
schema = json.loads(schema_path.read_text(encoding='utf-8'))
feature_keys = [f['key'] for f in schema.get('features', [])]

records = []
for r in rows:
  company = r.get('company', {}) or {}
  features = (r.get('features', {}) or {}).copy()
  for k in feature_keys:
    features.setdefault(k, 'unknown')
  records.append({
    'slug': company.get('slug', ''),
    'name': company.get('name', ''),
    'notes': r.get('notes', ''),
    **{k: features.get(k, 'unknown') for k in feature_keys}
  })

df = pd.DataFrame.from_records(records).sort_values('slug').reset_index(drop=True)
feature_cols = feature_keys

df.head(3)


,slug,name,notes,productType,collectionBucket,uxMode,imageryUsage,colorStrategy,typographyTone,layoutDensity,surfaceDepth,themeMode,shapeLanguage,shadowStyle,contentFocus,motionUsage,primaryIntent
0,airbnb,Airbnb,"Airbnb's design exudes a warm, inviting atmosp...",marketplace,enterpriseAndConsumer,browsingHeavy,imageFirst,singleAccent,expressive,balanced,layered,lightFirst,rounded,stacked,photography,medium,exploration
1,airtable,Airtable,Airtable's design exudes a vibe of sophisticat...,digital,designAndProductivity,browsingHeavy,mixed,multiAccent,neutral,balanced,layered,lightFirst,rounded,subtle,productScreenshots,medium,trust
2,apple,Apple,"Apple's design exudes a premium, cinematic vib...",physical,enterpriseAndConsumer,browsingHeavy,imageFirst,singleAccent,premium,sparse,flat,darkFirst,pill,subtle,photography,low,trust


In [7]:
# Sanity checks
print('unique slugs:', df['slug'].nunique())
print('missing slugs:', (df['slug'] == '').sum())

for c in feature_cols:
  print(c, df[c].value_counts(dropna=False).to_dict())


unique slugs: 58
missing slugs: 0
productType {'digital': 51, 'physical': 6, 'marketplace': 1}
collectionBucket {'developerToolsAndPlatforms': 14, 'aiAndMachineLearning': 12, 'designAndProductivity': 10, 'enterpriseAndConsumer': 7, 'infrastructureAndCloud': 6, 'automotiveAndMobility': 5, 'fintechAndCrypto': 4}
uxMode {'browsingHeavy': 25, 'mixed': 14, 'taskFocused': 12, 'creationTool': 6, 'docsHeavy': 1}
imageryUsage {'mixed': 44, 'imageFirst': 13, 'textFirst': 1}
colorStrategy {'multiAccent': 20, 'singleAccent': 19, 'monochrome': 12, 'gradientLed': 7}
typographyTone {'technical': 25, 'expressive': 18, 'premium': 9, 'neutral': 6}
layoutDensity {'balanced': 33, 'dense': 18, 'sparse': 7}
surfaceDepth {'flat': 31, 'layered': 21, 'subtle': 6}
themeMode {'lightFirst': 28, 'darkFirst': 22, 'dual': 8}
shapeLanguage {'pill': 21, 'sharp': 17, 'rounded': 13, 'mixed': 7}
shadowStyle {'none': 21, 'subtle': 20, 'stacked': 12, 'ring': 5}
contentFocus {'productScreenshots': 13, 'codeFirst': 13, 'mixe

In [8]:
# Crosstab helpers

def crosstab(a: str, b: str | None = None):
  if b:
    t = pd.crosstab(df[a], df[b])
  else:
    t = df[a].value_counts().to_frame('count')
  return t

crosstab('businessModel')


KeyError: 'businessModel'

In [ ]:
# Example: compare businessModel vs imageryUsage
crosstab('businessModel', 'imageryUsage')


imageryUsage,imageFirst,mixed,textFirst,unknown
businessModel,,,,
ai,1,10,1,0
devtools,1,16,0,1
ecommerce,1,0,0,0
fintech,0,5,0,0
marketplace,1,1,0,0
media,1,0,0,0
other,2,2,0,0
saas,1,8,0,1
social,1,0,0,0


In [ ]:
# Slice helper: get examples with notes

def examples(where: dict[str, str], limit: int = 10):
  q = df
  for k, v in where.items():
    q = q[q[k] == v]
  cols = ['slug', 'name', 'notes'] + [k for k in where.keys()]
  return q[cols].head(limit)

examples({'businessModel': 'devtools', 'colorStrategy': 'monochrome'}, limit=10)


,slug,name,notes,businessModel,colorStrategy
10,composio,Composio,Composio's design exudes a developer-centric v...,devtools,monochrome
13,expo,Expo,"Expo's design exudes a premium, polished vibe ...",devtools,monochrome
29,ollama,Ollama,"Ollama's design embodies radical minimalism, f...",devtools,monochrome
38,sanity,Sanity,"Sanity's design exudes a dark, structured vibe...",devtools,monochrome
47,vercel,Vercel,"Vercel's design system embodies a clean, techn...",devtools,monochrome
49,warp,Warp,"Warp's design exudes a warm, inviting atmosphe...",devtools,monochrome


In [ ]:
# Optional clustering (one-hot encode enum features)
# If scikit-learn isn't installed, install it in your environment:
#   pip install scikit-learn

from sklearn.cluster import KMeans

X = pd.get_dummies(df[feature_cols], prefix=feature_cols)

k = 5
km = KMeans(n_clusters=k, n_init='auto', random_state=7)
labels = km.fit_predict(X)

df_clustered = df.assign(cluster=labels)
df_clustered['cluster'].value_counts().sort_index()


cluster
0     4
1    25
2     8
3     7
4    10
Name: count, dtype: int64

In [ ]:
# Inspect clusters: most common feature values per cluster

def cluster_profile(cluster_id: int):
  sub = df_clustered[df_clustered['cluster'] == cluster_id]
  out = {}
  for c in feature_cols:
    out[c] = sub[c].value_counts().head(3).to_dict()
  return pd.Series(out)

cluster_profile(0)


productType                                    {'digital': 4}
businessModel                         {'saas': 3, 'media': 1}
uxMode                 {'taskFocused': 3, 'browsingHeavy': 1}
imageryUsage      {'mixed': 2, 'unknown': 1, 'imageFirst': 1}
colorStrategy            {'singleAccent': 3, 'monochrome': 1}
typographyTone              {'technical': 2, 'expressive': 2}
layoutDensity                     {'balanced': 3, 'dense': 1}
surfaceDepth                        {'layered': 3, 'flat': 1}
themeMode                   {'lightFirst': 2, 'darkFirst': 2}
shapeLanguage                         {'pill': 2, 'mixed': 2}
shadowStyle                         {'stacked': 3, 'none': 1}
contentFocus                        {'productScreenshots': 4}
motionUsage             {'low': 2, 'medium': 1, 'unknown': 1}
primaryIntent                      {'trust': 3, 'unknown': 1}
dtype: object

In [ ]:
# Save a CSV snapshot for ad-hoc exploration
OUT_CSV = REPO_ROOT / 'analysis' / 'features.csv'
df.to_csv(OUT_CSV, index=False)
print('Wrote', OUT_CSV)


Wrote /Users/kihyvr/Documents/design-extraction/analysis/features.csv


## Clustering & segmentation

Below we:
- One-hot encode the enum features
- Sweep `k` and pick a good value (silhouette score)
- Generate cluster summaries + representative companies

Notes:
- `tokenMaturity` is currently constant (`high` for all rows), so it carries no information for clustering and is dropped.


In [ ]:
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Drop constant columns for clustering
cluster_cols = [c for c in feature_cols if df[c].nunique() > 1]
cluster_cols

['productType',
 'businessModel',
 'uxMode',
 'imageryUsage',
 'colorStrategy',
 'typographyTone',
 'layoutDensity',
 'surfaceDepth',
 'themeMode',
 'shapeLanguage',
 'shadowStyle',
 'contentFocus',
 'motionUsage',
 'primaryIntent']

In [ ]:
X = pd.get_dummies(df[cluster_cols])

# KMeans likes continuous space; standardize the one-hot matrix
Xs = StandardScaler(with_mean=False).fit_transform(X)

scores = []
for k in range(2, 11):
  km = KMeans(n_clusters=k, n_init='auto', random_state=7)
  labels = km.fit_predict(Xs)
  s = silhouette_score(Xs, labels)
  scores.append({'k': k, 'silhouette': s})

score_df = pd.DataFrame(scores).sort_values('silhouette', ascending=False)
score_df

,k,silhouette
0,2,0.344874
8,10,0.041241
1,3,0.038887
7,9,0.029517
5,7,0.026329
3,5,0.019426
6,8,0.018614
4,6,0.005837
2,4,0.004287


In [ ]:
# Pick k (you can override this manually)
best_k = int(score_df.iloc[0]['k'])
best_k

2

In [ ]:
km = KMeans(n_clusters=best_k, n_init='auto', random_state=7)
labels = km.fit_predict(Xs)

df_clustered = df.assign(cluster=labels)
df_clustered['cluster'].value_counts().sort_index()

cluster
0    53
1     1
Name: count, dtype: int64

In [ ]:
# 2D projection for quick visual sanity check (not a "truth")
pca = PCA(n_components=2, random_state=7)
X2 = pca.fit_transform(Xs.toarray() if hasattr(Xs, 'toarray') else Xs)

plot_df = pd.DataFrame({
  'x': X2[:, 0],
  'y': X2[:, 1],
  'cluster': labels,
  'slug': df['slug'],
  'businessModel': df['businessModel'],
  'uxMode': df['uxMode']
})
plot_df.head()

,x,y,cluster,slug,businessModel,uxMode
0,-0.851094,7.813679,0,airbnb,marketplace,browsingHeavy
1,-2.940367,0.153773,0,airtable,saas,browsingHeavy
2,5.539464,4.162196,0,apple,ecommerce,browsingHeavy
3,6.442264,3.103026,0,bmw,other,browsingHeavy
4,0.010803,-1.208934,0,cal,saas,taskFocused


In [ ]:
# Cluster summaries: dominant feature values + representative companies

def top_values(series: pd.Series, n: int = 3):
  vc = series.value_counts()
  total = vc.sum()
  return [f"{idx} ({cnt}/{total})" for idx, cnt in vc.head(n).items()]

summaries = []
for cid, sub in df_clustered.groupby('cluster'):
  row = {'cluster': cid, 'n': len(sub)}
  for c in cluster_cols:
    row[c] = '; '.join(top_values(sub[c], n=3))
  summaries.append(row)

summary_df = pd.DataFrame(summaries).sort_values('n', ascending=False)
summary_df

,cluster,n,productType,businessModel,uxMode,imageryUsage,colorStrategy,typographyTone,layoutDensity,surfaceDepth,themeMode,shapeLanguage,shadowStyle,contentFocus,motionUsage,primaryIntent
0,0,53,digital (49/53); physical (2/53); marketplace ...,devtools (18/53); ai (12/53); saas (10/53),browsingHeavy (20/53); mixed (14/53); taskFocu...,mixed (42/53); imageFirst (8/53); unknown (2/53),multiAccent (20/53); singleAccent (15/53); mon...,technical (25/53); expressive (15/53); premium...,balanced (30/53); dense (17/53); sparse (6/53),flat (28/53); layered (19/53); subtle (6/53),lightFirst (27/53); darkFirst (20/53); dual (6...,pill (20/53); rounded (13/53); sharp (13/53),subtle (17/53); none (17/53); stacked (10/53),codeFirst (13/53); productScreenshots (12/53);...,low (30/53); medium (23/53),trust (39/53); exploration (7/53); emotionalBr...
1,1,1,digital (1/1),media (1/1),browsingHeavy (1/1),imageFirst (1/1),singleAccent (1/1),expressive (1/1),dense (1/1),layered (1/1),darkFirst (1/1),pill (1/1),stacked (1/1),productScreenshots (1/1),unknown (1/1),unknown (1/1)


In [ ]:
def representatives(cluster_id: int, n: int = 8):
  sub = df_clustered[df_clustered['cluster'] == cluster_id]
  cols = ['slug', 'name', 'notes'] + cluster_cols
  return sub[cols].head(n)

representatives(int(summary_df.iloc[0]['cluster']), n=10)

,slug,name,notes,productType,businessModel,uxMode,imageryUsage,colorStrategy,typographyTone,layoutDensity,surfaceDepth,themeMode,shapeLanguage,shadowStyle,contentFocus,motionUsage,primaryIntent
0,airbnb,Airbnb,"Airbnb's design exudes a warm, inviting atmosp...",marketplace,marketplace,browsingHeavy,imageFirst,singleAccent,expressive,balanced,layered,lightFirst,rounded,stacked,photography,medium,exploration
1,airtable,Airtable,Airtable's design exudes a vibe of sophisticat...,digital,saas,browsingHeavy,mixed,multiAccent,neutral,balanced,layered,lightFirst,rounded,subtle,productScreenshots,medium,trust
2,apple,Apple,"Apple's design exudes a premium, cinematic vib...",physical,ecommerce,browsingHeavy,imageFirst,singleAccent,premium,sparse,flat,darkFirst,pill,subtle,photography,low,trust
3,bmw,BMW,BMW's design system exudes precision and perfo...,physical,other,browsingHeavy,imageFirst,singleAccent,technical,dense,flat,darkFirst,sharp,none,photography,low,trust
4,cal,Cal.com,"Cal.com embodies a clean, professional aesthet...",digital,saas,taskFocused,unknown,monochrome,technical,balanced,layered,lightFirst,pill,stacked,productScreenshots,low,trust
5,claude,Claude,"Claude's design exudes a warm, intellectual vi...",digital,ai,browsingHeavy,mixed,multiAccent,expressive,balanced,layered,dual,rounded,ring,illustration,low,emotionalBranding
6,clay,Clay,"Clay's design exudes a warm, playful vibe, uti...",digital,saas,mixed,mixed,multiAccent,expressive,balanced,layered,lightFirst,mixed,stacked,mixed,medium,emotionalBranding
7,clickhouse,ClickHouse,"ClickHouse's design exudes a high-performance,...",digital,devtools,browsingHeavy,mixed,singleAccent,technical,dense,layered,darkFirst,sharp,unknown,codeFirst,low,trust
8,cohere,Cohere,"Cohere's design exudes a polished, enterprise-...",digital,ai,taskFocused,mixed,monochrome,technical,balanced,flat,lightFirst,mixed,none,productScreenshots,low,trust
9,coinbase,Coinbase,Coinbase's design exudes a clean and trustwort...,digital,fintech,browsingHeavy,mixed,singleAccent,neutral,balanced,flat,lightFirst,pill,none,productScreenshots,low,trust


In [ ]:
# Save cluster assignments for later (can be imported to Neo4j too)
OUT_CLUSTER_CSV = REPO_ROOT / 'analysis' / 'clusters.csv'
df_clustered[['slug', 'cluster']].to_csv(OUT_CLUSTER_CSV, index=False)
print('Wrote', OUT_CLUSTER_CSV)

Wrote /Users/kihyvr/Documents/design-extraction/analysis/clusters.csv
